In [1]:
# 单元格 1：配置和导入
from pathlib import Path
import sys
root_dir = Path.cwd().parent.parent
sys.path.append(str(root_dir))

import os
from dotenv import load_dotenv
from globalobjects.json_manager import JSONManager

load_dotenv(os.getenv('ENV_FILE', os.path.join(root_dir, '.env')))
PROJECT_DIR = os.getenv("PROJECT_DIR")
CACHE_FILENAME = os.getenv("CACHE_FILENAME")
CACHE_FILE = JSONManager(f"{root_dir}/project_files/{PROJECT_DIR}/{CACHE_FILENAME}")

MYAPS_MAIN_DB = CACHE_FILE.get('env').get("MYAPS_MAIN_DB", "default")
MYAPS_DB_HOST = CACHE_FILE.get('env').get("MYAPS_DB_HOST", "localhost")
MYAPS_DB_PORT = CACHE_FILE.get('env').get("MYAPS_DB_PORT", 3333)
MYAPS_DB_USER = CACHE_FILE.get('env').get("MYAPS_DB_USER", "root")
MYAPS_DB_PASSWORD = CACHE_FILE.get('env').get("MYAPS_DB_PASSWORD", "123456")

connections = {
    MYAPS_MAIN_DB: {
        "engine": "tortoise.backends.mysql",
        "credentials": {
            "host": MYAPS_DB_HOST,
            "port": MYAPS_DB_PORT,
            "user": MYAPS_DB_USER,
            "password": MYAPS_DB_PASSWORD,
            "database": MYAPS_MAIN_DB,
            "charset": "utf8mb4",
            "connect_timeout": 5,
        }
    }
}

# 单元格 2：初始化 Tortoise（包含状态重置）
from apps.io_api.models import TMaterial
from tortoise import Tortoise

TORTOISE_ORM_CONFIG = {
    "connections": connections,
    "apps": {
        "io_api_models": {
            "models": ["apps.io_api.models"],
            "default_connection": MYAPS_MAIN_DB
        },
    },
}

# 重置状态
if Tortoise._inited:
    await Tortoise.close_connections()
    Tortoise._inited = False
    print("已重置 Tortoise 状态")

# 初始化
await Tortoise.init(config=TORTOISE_ORM_CONFIG)
print("Tortoise ORM 初始化完成")



2026-03-27 19:19:18 - WARNING - ⚠️ 环境变量配置：MYAPS_DB_SET 未设置
Tortoise ORM 初始化完成


In [ ]:
# 单元格 3：执行查询
result = await TMaterial.filter(materialno="01000").using_db(Tortoise.get_connection(MYAPS_MAIN_DB)).values('description', 'materialno')
print(result)

[{'description': 'cs成品', 'materialno': '01000'}]


In [9]:
import asyncio
from apps.io_api.models import TMaterial
from tortoise import Tortoise
# from settings import MYAPS_MAIN_DB

m = await TMaterial.get(materialno="01000").using_db(Tortoise.get_connection(MYAPS_MAIN_DB)).values('free1')
print(m)
print(type(m))

{'free1': None}
<class 'dict'>
